In [1]:
# 1 — Import libraries and load data

import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "notebooks" else cwd

DATA_PATH = PROJECT_ROOT / "data" / "cleaned" / "adult_cleaned.csv"

df = pd.read_csv(DATA_PATH, dtype={"zip_code": "string"})

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (32561, 16)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income,zip_code
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K,93810
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K,24592
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K,13278
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K,46048
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,42098


In [2]:
# 2 — Separate features and target

TARGET = "income"

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found.")

# Remove records with a missing target
df = df.dropna(subset=[TARGET]).copy()

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(str).str.strip()

# Ensure ZIP code is handled as a category
if "zip_code" in X.columns:
    X["zip_code"] = X["zip_code"].astype("string")

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("\nTarget distribution:")
display(y.value_counts())

Numeric features: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
Categorical features: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country', 'zip_code']

Target distribution:


income
<=50K    24720
>50K      7841
Name: count, dtype: int64

In [3]:
# 3 — Split into training and testing sets  

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

print("\nTraining income proportions:")
display(y_train.value_counts(normalize=True).round(3))

print("\nTesting income proportions:")
display(y_test.value_counts(normalize=True).round(3))

Training features: (26048, 15)
Testing features: (6513, 15)
Training targets: (26048,)
Testing targets: (6513,)

Training income proportions:


income
<=50K    0.759
>50K     0.241
Name: proportion, dtype: float64


Testing income proportions:


income
<=50K    0.759
>50K     0.241
Name: proportion, dtype: float64

In [4]:
# 4 — Build the preprocessing pipeline

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


In [5]:
# 5 — Fit on training data only 

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

assert X_train_processed.shape[1] == X_test_processed.shape[1]
print("Validation passed: both datasets have the same feature count.")

Processed training shape: (26048, 22716)
Processed testing shape: (6513, 22716)
Validation passed: both datasets have the same feature count.


In [8]:
# 6 — Validate the pipeline

assert X_train_processed.shape[0] == len(y_train)
assert X_test_processed.shape[0] == len(y_test)

assert not pd.isna(y_train).any()
assert not pd.isna(y_test).any()

print("All pipeline validations passed!")
print("Training features:", X_train_processed.shape)
print("Testing features:", X_test_processed.shape)

All pipeline validations passed!
Training features: (26048, 22716)
Testing features: (6513, 22716)


In [9]:
# 7 — Save the preprocessing metadata

report_dir = PROJECT_ROOT / "report"
report_dir.mkdir(parents=True, exist_ok=True)

metadata = f"""# Day 3: Preprocessing Pipeline

- Dataset: Adult Census Income (cleaned)
- Target: income
- Random seed: 42
- Train/test split: 80/20
- Stratification: income
- Numeric features: {numeric_features}
- Categorical features: {categorical_features}
- Training rows: {X_train.shape[0]}
- Testing rows: {X_test.shape[0]}
- Processed training shape: {X_train_processed.shape}
- Processed testing shape: {X_test_processed.shape}

Preprocessing was fitted on training data only.
The test set was transformed using the fitted pipeline.
ZIP-code provenance must be documented separately.
"""

metadata_path = report_dir / "Day3_Preprocessing_Summary.txt"
metadata_path.write_text(metadata, encoding="utf-8")

print("Saved:", metadata_path)

Saved: d:\VS Code\Projects\python\Internship\report\Day3_Preprocessing_Summary.txt
